# Lab 9.5 &mdash; The No-Advice Guardrail

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 5 &middot; Module 9 &mdash; Agents in Finance, Healthcare &amp; Cybersecurity**

### What you'll do
- Detect buy/sell/recommend language in a summary
- Reject any output that crosses into advice
- Keep the agent informational -- analysis, not a recommendation

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it** (the real grounding / citation / compute logic, or the real `create_agent` wiring), then **Run it** and read the output &mdash; and, for the agent labs, the real **message trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working, grounded insight agent you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`). The insight agent is a **real** `create_agent` driven by a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")`, key in `.env` as `GROQ_API_KEY`). You are building the **financial-report insight agent** &mdash; the client's Lab 5.1. It is **informational only**: it grounds &amp; **cites** every figure, gives **no advice**, and has **no trade tool** (`place_trade` is defined but never bound &mdash; a human analyst decides). A `@tool` must **catch its own errors and return a string** &mdash; a tool that raises can abort the whole run. If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing.

**Reference:** [Module 9 slides &mdash; The insight agent, in code](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 9 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv
load_dotenv(pathlib.Path("/home/rajesh/Training/courses/building-intelligents-ai-agents/.env"), override=True)   # GROQ_API_KEY (the Day-5 provider)

WORK = "/tmp/biaa-lab-09-05"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is set. Day-5 labs call a REAL hosted model (Groq)."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-5 provider: a REAL hosted model. openai/gpt-oss-20b is a reliable tool-caller via create_agent.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY set | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
The insight agent has a hard boundary (deck slides 6, 9): it **informs** &mdash; it does **not** give
personalized investment advice. Providing buy/sell recommendations is a regulated activity, out of scope,
and dangerous to hand to an LLM. So a guardrail **detects advice language** and rejects it: the output stays
**analysis**, and a human analyst draws any conclusion. In Lab 10 you'll run this same check over a **real
model's** output.

In [ ]:
# Words that turn analysis into (forbidden) advice.
ADVICE_TERMS = ("buy", "sell", "you should", "recommend", "strong buy", "invest in")
print("advice terms to block:", ADVICE_TERMS)

## Build it
Complete `contains_advice` and `enforce_no_advice` (reject a summary that gives advice), then run the
cell.

In [ ]:
ADVICE_TERMS = ("buy", "sell", "you should", "recommend", "strong buy", "invest in")

def contains_advice(text):
    t = text.lower()
    return ___   # TODO: True if ANY advice term appears in the text

def enforce_no_advice(summary):
    # a valid insight is informational; reject it if it crosses into advice
    return {"ok": ___, "summary": summary}   # TODO: ok = it does NOT contain advice

try:
    clean  = "Revenue +12% YoY [p4]; net margin down to 7.5% [p4] -- FLAG."
    advice = "Revenue is up -- you should buy this stock now."
    print("clean  ->", enforce_no_advice(clean))
    print("advice ->", enforce_no_advice(advice))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- A clean, cited summary passes; the moment it says "you should buy", it's **rejected** &mdash; informational only.
- This is a deterministic guardrail you own; it runs *after* the model, on whatever text the model produced.
- Withholding the trade tool (Lab 6) stops the agent *acting*; this stops it *advising*. Both are needed.

## Your turn (open task &mdash; no grader)
The keyword list is a blunt instrument &mdash; try a phrasing that slips past it (e.g. *"a compelling
entry point"*) and decide whether you'd add it. **What good looks like:** you understand the guardrail's
limits and can tighten it, while accepting that the *real* safety net is the withheld tool, not the word
list.

---
### Key takeaway
The agent informs, never advises. Detecting and blocking advice language keeps it on the safe, valuable side of the line -- analysis for a human, not a recommendation. Next: the stronger guardrail.

[&#8592; All Module 9 labs](./index.html) &nbsp;&middot;&nbsp; [Module 9 slides](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>